In [1]:
from Simulation.mpc import *
from Simulation.system_functions import PolymerCSTR
from utils.helpers import *

## Initialize the system

In [2]:
# First initiate the system
# Parameters
Ad = 2.142e17           # h^-1
Ed = 14897              # K
Ap = 3.816e10           # L/(molh)
Ep = 3557               # K
At = 4.50e12            # L/(molh)
Et = 843                # K
fi = 0.6                # Coefficient
m_delta_H_r = -6.99e4   # j/mol
hA = 1.05e6             # j/(Kh)
rhocp = 1506            # j/(Kh)
rhoccpc = 4043          # j/(Kh)
Mm = 104.14             # g/mol
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

In [3]:
# Design Parameters
CIf = 0.5888    # mol/L
CMf = 8.6981    # mol/L
Qi = 108.       # L/h
Qs = 459.       # L/h
Tf = 330.       # K
Tcf = 295.      # K
V = 3000.       # L
Vc = 3312.4     # L

system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

In [4]:
# Steady State Inputs
Qm_ss = 378.    # L/h
Qc_ss = 471.6   # L/h

system_steady_state_inputs = np.array([Qc_ss, Qm_ss])

In [5]:
# Sampling time of the system
delta_t = 0.5 # 30 mins

In [6]:
# Initiate the CSTR for steady state values
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states={"ss_inputs":cstr.ss_inputs,
               "y_ss":cstr.y_ss}

## Loading the system matrices, min max scaling, and min max of the states

In [7]:
dir_path = os.path.join(os.getcwd(), "Data")

In [8]:
# Defining the range of setpoints for data generation
setpoint_y = np.array([[3.2, 321],
                       [4.5, 325]])
u_min = np.array([71.6, 78])
u_max = np.array([870, 670])

system_data = load_and_prepare_system_data(steady_states=steady_states, setpoint_y=setpoint_y, u_min=u_min, u_max=u_max)

In [9]:
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]

In [10]:
data_min = system_data["data_min"]
data_max = system_data["data_max"]

In [11]:
min_max_states = {'max_s': np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ]),
                  'min_s': np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])}

In [12]:
y_sp_scaled_deviation = system_data["y_sp_scaled_deviation"]

In [13]:
b_min = system_data["b_min"]
b_max = system_data["b_max"]

In [14]:
min_max_dict = system_data["min_max_dict"]
min_max_dict["x_max"] = np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ])
min_max_dict["x_min"] = np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])

In [15]:
# Setpoints in deviation form
inputs_number = int(B_aug.shape[1])
y_sp_scenario = np.array([[4.5, 324],
                          [3.4, 321]])

y_sp_scenario = (apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:])
                 - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:]))
n_tests = 200
set_points_len = 400
TEST_CYCLE = [False, False, False, False, False]
warm_start = 5
ACTOR_FREEZE = 0 * set_points_len
warm_start_plot = warm_start * 2 * set_points_len + ACTOR_FREEZE

In [16]:
# Observer Gain
poles = np.array(np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966,
                           0.42900673, 0.4228262 , 0.96916776, 0.91230187]))
L = compute_observer_gain(A_aug, C_aug, poles)

The system is observable.


C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Simulation\mpc.py:123: UserWarning: Convergence was not reached after maxiter iterations.
You asked for a tolerance of 0.001, we got 0.9999999422182038.
  obs_gain_calc = signal.place_poles(A.T, C.T, desired_poles, method='KNV0')


# DQN network

In [17]:
from DQN.dqn_agent import DQNAgent
import torch

In [18]:
PREDICT_GRID = list(range(8, 20))  # Hp candidates
CONTROL_GRID = list(range(3, 10))     # Hc candidates (will be filtered by Hc <= Hp)

In [19]:
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
N_ACTIONS = len(HORIZON_RECIPES)

In [20]:
STATE_DIM = int(A_aug.shape[0]) + int(C_aug.shape[0]) + int(B_aug.shape[1])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
HIDDEN_LAYERS = [512, 512, 512, 512, 512]
BUFFER_SIZE = 40_000
DECISION_INTERVAL = 4
INPUT_NUMBER = B_aug.shape[1]

Using device: cuda


In [21]:
dqn_agent = DQNAgent(
    state_dim=STATE_DIM,
    action_dim=N_ACTIONS,
    hidden_dim=HIDDEN_LAYERS,
    gamma=0.99,
    lr=1e-4,
    batch_size=128,
    buffer_size=BUFFER_SIZE,
    grad_clip_norm=10.0,
    double_dqn=True,
    target_update="soft",
    tau=0.01,
    hard_update_interval=10_000,
    target_combine="q1",
    activation="relu",
    use_layer_norm=False,
    dropout=0.0,
    device=DEVICE,
    eps_start=0.3,
    eps_end=0.01,
    eps_decay_rate=0.99999,
    eps_decay_mode="exp",
)

In [22]:
# MPC parameters
n_inputs = 2
predict_h = 9
cont_h = 3
u_ss = apply_min_max(steady_states['ss_inputs'], data_min[:n_inputs], data_max[:n_inputs])
b_min = apply_min_max(np.array([71.6, 78]), data_min[:n_inputs], data_max[:n_inputs])
b_max= apply_min_max(np.array([870, 670]), data_min[:n_inputs], data_max[:n_inputs])
b1 = (b_min[0]-u_ss[0], b_max[0]-u_ss[0])
b2 = (b_min[1]-u_ss[1], b_max[1]-u_ss[1])
bnds = (b1, b2)*cont_h
cons = []
IC_opt = np.zeros(n_inputs*cont_h)
Q1_penalty = 5.
Q2_penalty = 1.
R1_penalty = 1
R2_penalty = 1

In [32]:
def make_reward_fn_relative_QR(
    data_min, data_max, n_inputs,
    k_rel, band_floor_phys,
    Q_diag, R_diag,
    tau_frac=0.7,
    gamma_out=0.5, gamma_in=0.5,
    beta=5.0, gate="geom", lam_in=1.0,
    bonus_kind="exp", bonus_k=12.0, bonus_p=0.6, bonus_c=20.0,
):
    """
    Reward with relative tracking bands.

    data_min, data_max : arrays for [u_min..., y_min...], [u_max..., y_max...]
    n_inputs           : number of inputs (so outputs start at index n_inputs)
    k_rel              : per-output relative tolerance factors (same length as outputs)
    band_floor_phys    : per-output minimum band in physical units
    Q_diag, R_diag     : quadratic weights (same as before)
    """

    data_min = np.asarray(data_min, float)
    data_max = np.asarray(data_max, float)
    dy = np.maximum(data_max[n_inputs:] - data_min[n_inputs:], 1e-12)  # phys range for each y

    k_rel = np.asarray(k_rel, float)
    band_floor_phys = np.asarray(band_floor_phys, float)
    Q_diag = np.asarray(Q_diag, float)
    R_diag = np.asarray(R_diag, float)

    # floor in *scaled* coordinates (used if y_sp_phys is not provided)
    band_floor_scaled = band_floor_phys / np.maximum(dy, 1e-12)

    def _sigmoid(x):
        x = np.clip(x, -60.0, 60.0)
        return 1.0 / (1.0 + np.exp(-x))

    def _phi(z, kind=bonus_kind, k=bonus_k, p=bonus_p, c=bonus_c):
        z = np.clip(z, 0.0, 1.0)
        if kind == "linear":
            return 1.0 - z
        if kind == "quadratic":
            return (1.0 - z) ** 2
        if kind == "exp":
            return (np.exp(-k * z) - np.exp(-k)) / (1.0 - np.exp(-k))
        if kind == "power":
            return 1.0 - np.power(z, p)
        if kind == "log":
            return np.log1p(c * (1.0 - z)) / np.log1p(c)
        raise ValueError("unknown bonus kind")

    def reward_fn(e_scaled, du_scaled, y_sp_phys=None):
        """
        e_scaled : output error in scaled deviation space  (same as before)
        du_scaled: input move in scaled deviation space    (same as before)
        y_sp_phys: current setpoint in *physical* units (array len = n_outputs)
        """

        e_scaled = np.asarray(e_scaled, float)
        du_scaled = np.asarray(du_scaled, float)

        # ----- dynamic band based on setpoint -----
        if y_sp_phys is None:
            # fallback: just use the floor
            band_scaled = band_floor_scaled
        else:
            y_sp_phys_arr = np.asarray(y_sp_phys, float)
            # band_phys_i = max(k_rel_i * |y_sp_i|, band_floor_phys_i)
            band_phys = np.maximum(k_rel * np.abs(y_sp_phys_arr), band_floor_phys)
            band_scaled = band_phys / np.maximum(dy, 1e-12)

        tau_scaled = tau_frac * band_scaled

        # ----- inside/outside gate -----
        abs_e = np.abs(e_scaled)
        s_i = _sigmoid((band_scaled - abs_e) / np.maximum(tau_scaled, 1e-12))

        if gate == "prod":
            w_in = float(np.prod(s_i, dtype=np.float64))
        elif gate == "mean":
            w_in = float(np.mean(s_i))
        elif gate == "geom":
            w_in = float(np.prod(s_i, dtype=np.float64) ** (1.0 / len(s_i)))
        else:
            raise ValueError("gate must be 'prod'|'mean'|'geom'")

        # ----- core quadratic costs -----
        err_quad = np.sum(Q_diag * (e_scaled ** 2))
        err_eff = (1.0 - w_in) * err_quad + w_in * (lam_in * err_quad)
        move = np.sum(R_diag * (du_scaled ** 2))

        # ----- linear penalties around band edge -----
        slope_at_edge = 2.0 * Q_diag * band_scaled

        overflow = np.maximum(abs_e - band_scaled, 0.0)
        lin_out = (1.0 - w_in) * np.sum(gamma_out * slope_at_edge * overflow)

        inside_mag = np.minimum(abs_e, band_scaled)
        lin_in = w_in * np.sum(gamma_in * slope_at_edge * inside_mag)

        # ----- bonus near zero error -----
        qb2 = Q_diag * (band_scaled ** 2)
        z = abs_e / np.maximum(band_scaled, 1e-12)
        phi = _phi(z)
        bonus = w_in * beta * np.sum(qb2 * phi)

        # ----- total reward -----
        return (-(err_eff + move + lin_out + lin_in) + bonus) * 0.01

    params = dict(
        k_rel=k_rel,
        band_floor_phys=band_floor_phys,
        band_floor_scaled=band_floor_scaled,
        Q_diag=Q_diag,
        R_diag=R_diag,
        tau_frac=tau_frac,
        gamma_out=gamma_out,
        gamma_in=gamma_in,
        beta=beta,
        gate=gate,
        lam_in=lam_in,
        bonus_kind=bonus_kind,
        bonus_k=bonus_k,
        bonus_p=bonus_p,
        bonus_c=bonus_c,
    )
    return params, reward_fn

In [33]:
n_inputs = 2

dy = data_max[n_inputs:] - data_min[n_inputs:]
y_sp_nom = 0.5 * (data_min[n_inputs:] + data_max[n_inputs:])

k_rel = np.array([0.003, 0.0003])
band_floor_phys = np.array([0.006, 0.07])

band_phys = np.maximum(k_rel * np.abs(y_sp_nom), band_floor_phys)

scale_factor = 1.0  # use 2.0 for [-1, 1] scaling, 1.0 for [0, 1]
band_scaled = scale_factor * band_phys / dy

q0 = 1.4
Q_diag = q0 / np.maximum(band_scaled ** 2, 1e-12)

print("dy:", dy)
print("y_sp_nom:", y_sp_nom)
print("band_phys:", band_phys)
print("band_scaled:", band_scaled)
print("Q_diag:", Q_diag)

dy: [0.22165278 0.78153727]
y_sp_nom: [  3.83915067 323.21371982]
band_phys: [0.01151745 0.09696412]
band_scaled: [0.05196169 0.12406845]
Q_diag: [518.51529284  90.95055189]


In [34]:
Q_diag = np.array([518., 90.])          # rounded from the band-based calculation
R_diag = np.array([90., 90.])          # move cost for du_scaled ~ 0.02

n_inputs = 2

print("Band scaled are:")

params, reward_fn = make_reward_fn_relative_QR(
    data_min, data_max, n_inputs,
    k_rel, band_floor_phys,
    Q_diag, R_diag,
    tau_frac=0.7,
    gamma_out=0.5, gamma_in=0.5,
    beta=7.0, gate="geom", lam_in=1.0,
    bonus_kind="exp", bonus_k=12.0, bonus_p=0.6, bonus_c=20.0,
)
print(params)

Band scaled are:
{'k_rel': array([0.003 , 0.0003]), 'band_floor_phys': array([0.006, 0.07 ]), 'band_floor_scaled': array([0.02706937, 0.08956707]), 'Q_diag': array([518.,  90.]), 'R_diag': array([90., 90.]), 'tau_frac': 0.7, 'gamma_out': 0.5, 'gamma_in': 0.5, 'beta': 7.0, 'gate': 'geom', 'lam_in': 1.0, 'bonus_kind': 'exp', 'bonus_k': 12.0, 'bonus_p': 0.6, 'bonus_c': 20.0}


In [26]:
nominal_qs = 459
nominal_qi = 108
nominal_hA = 1.05e6
qi_change = 0.85
qs_change = 1.3
ha_change = 0.85

In [27]:
def run_dqn_mpc_horizon_supervisor(system, y_sp_scenario, n_tests, set_points_len,
                 steady_states, min_max_dict, agent, A_aug, B_aug, C_aug,
                 Q1_penalty, Q2_penalty, R1_penalty, R2_penalty, L,
                 data_min, data_max, warm_start, test_cycle,
                 h_recipes,
                 nominal_qi, nominal_qs, nominal_ha,
                 qi_change, qs_change, ha_change,
                 b_min, b_max, reward_fn, mode="disturb", predict_h = 6, cont_h = 3):

    # --- setpoints generation ---
    y_sp, nFE, sub_episodes_changes_dict, time_in_sub_episodes, test_train_dict, WARM_START, qi, qs, ha = \
        generate_setpoints_training_rl_gradually(
            y_sp_scenario, n_tests, set_points_len, warm_start, test_cycle,
            nominal_qi, nominal_qs, nominal_ha,
            qi_change, qs_change, ha_change
        )

    # inputs and outputs of the system dimensions
    n_inputs = B_aug.shape[1]
    n_outputs = C_aug.shape[0]
    n_states = A_aug.shape[0]

    # Scaled steady states inputs and outputs
    ss_scaled_inputs = apply_min_max(steady_states["ss_inputs"], data_min[:n_inputs], data_max[:n_inputs])
    y_ss_scaled = apply_min_max(steady_states["y_ss"], data_min[n_inputs:], data_max[n_inputs:])

    # Logging
    y_system = np.zeros((nFE + 1, n_outputs))
    y_system[0, :] = system.current_output
    u_mpc = np.zeros((nFE, n_inputs))
    rewards = np.zeros(nFE)
    avg_rewards = []
    yhat = np.zeros((n_outputs, nFE))
    xhatdhat = np.zeros((n_states, nFE + 1))
    horizon_trace = np.zeros((nFE, 2), dtype=int)

    # --- MPC object: start with some default recipe ---
    last_a = None
    current_Hp, current_Hc = (predict_h, cont_h)
    MPC_obj = MpcSolverGeneral(A_aug, B_aug, C_aug,
            Q_out=np.array([Q1_penalty, Q2_penalty]),
            R_in=np.array([R1_penalty, R2_penalty]),
            NP=predict_h,
            NC=cont_h)

    # Bounds and constraints
    b1 = (b_min[0], b_max[0])
    b2 = (b_min[1], b_max[1])
    cons = []

    # Helper to rebuild solver when Hc/Hp change
    def rebuild_mpc(Hp: int, Hc: int):
        nonlocal MPC_obj
        MPC_obj = MpcSolverGeneral(A_aug, B_aug, C_aug,
                    Q_out=np.array([Q1_penalty, Q2_penalty]),
                    R_in=np.array([R1_penalty, R2_penalty]),
                    NP=Hp,
                    NC=Hc)

    a_idx_mpc = [i for i , j in enumerate(h_recipes) if j == (predict_h, cont_h)][0]

    for i in range(nFE):
        # Identify test or train phase
        # train/test phase
        if i in test_train_dict:
            test = test_train_dict[i]

        # Current scaled input & deviation
        scaled_current_input = apply_min_max(system.current_input, data_min[:n_inputs], data_max[:n_inputs])
        scaled_current_input_dev = scaled_current_input - ss_scaled_inputs

        # ---- DQN state ----
        current_rl_state = apply_rl_scaled(min_max_dict, xhatdhat[:, i], y_sp[i, :], scaled_current_input_dev)

        # ---- Choose action (horizons) occasionally ----
        if i > WARM_START:
            if (i % DECISION_INTERVAL == 0) or (last_a is None):
                if not test:
                    a_idx = agent.take_action(current_rl_state.astype(np.float32), eval_mode=test)
                else:
                    a_idx = agent.act_eval(current_rl_state.astype(np.float32))
                Hp, Hc = action_to_horizons(h_recipes, a_idx)
                # Rebuild MPC only if changed
                if (Hp, Hc) != (current_Hp, current_Hc):
                    rebuild_mpc(Hp, Hc)
                    current_Hp, current_Hc = Hp, Hc
                last_a = a_idx
            else:
                a_idx = last_a
                Hp, Hc = current_Hp, current_Hc
        else:
            a_idx = a_idx_mpc
            Hp, Hc = action_to_horizons(h_recipes, a_idx)

        horizon_trace[i] = (Hp, Hc)

        # ---- Solve MPC with current horizons ----
        # Bounds and initial guess must match current Hc
        bnds  = (b1, b2) * Hc
        IC_opt = np.zeros(n_inputs * Hc)

        sol = spo.minimize(
            lambda x: MPC_obj.mpc_opt_fun(x, y_sp[i, :], scaled_current_input_dev, xhatdhat[:, i]),
            IC_opt, bounds=bnds, constraints=cons
        )

        # Take first move (scaled deviation) and map to plant input
        u_mpc[i, :] = sol.x[:n_inputs] + ss_scaled_inputs
        u_plant = reverse_min_max(u_mpc[i, :], data_min[:n_inputs], data_max[:n_inputs])

        # delta u cost variables
        delta_u = u_mpc[i, :] - scaled_current_input

        # Apply to plant and step
        system.current_input = u_plant
        system.step()
        if mode == "disturb":
            # disturbances
            system.hA = ha[i]
            system.Qs = qs[i]
            system.Qi = qi[i]

        # Record output
        y_system[i+1, :] = system.current_output

        # ----- Observer & model roll -----
        y_current_scaled = apply_min_max(y_system[i+1, :], data_min[n_inputs:], data_max[n_inputs:]) - y_ss_scaled
        y_prev_scaled = apply_min_max(y_system[i, :], data_min[n_inputs:], data_max[n_inputs:]) - y_ss_scaled

        # Calculate Delta y in deviation form
        delta_y = y_current_scaled - y_sp[i, :]

        # Calculate the next state in deviation form
        yhat[:, i] = np.dot(MPC_obj.C, xhatdhat[:, i])
        xhatdhat[:, i+1] = np.dot(MPC_obj.A, xhatdhat[:, i]) + np.dot(MPC_obj.B, (u_mpc[i, :] - ss_scaled_inputs)) + np.dot(L, (y_prev_scaled - yhat[:, i])).T

        # y_sp in physical band
        y_sp_phys = reverse_min_max(y_sp[i, :] + y_ss_scaled, data_min[n_inputs:], data_max[n_inputs:])

        # Reward Calculation
        reward = reward_fn(delta_y, delta_u, y_sp_phys) * 0.01

        # Record rewards
        rewards[i] = reward

        # ----- Next state for DQN -----
        next_u_dev = u_mpc[i, :] - ss_scaled_inputs
        next_rl_state = apply_rl_scaled(min_max_dict, xhatdhat[:, i+1], y_sp[i, :], next_u_dev)

        # Episode boundary (treat each setpoint block as an episode end)
        done = 0.0

        # Buffer + train (skip if in test phase)
        if not test:
            if i > time_in_sub_episodes:
                agent.push(current_rl_state.astype(np.float32),
                           int(a_idx),
                           float(reward),
                           next_rl_state.astype(np.float32),
                           float(done))
            if i >= warm_start:
                _ = agent.train_step()  # returns loss or None

        # Print diagnostics on sub-episode boundary
        if i in sub_episodes_changes_dict:
            avg_rewards.append(np.mean(rewards[max(0, i - time_in_sub_episodes + 1): i + 1]))
            print('Sub_Episode:', sub_episodes_changes_dict[i], '| avg. reward:', avg_rewards[-1],
                  '| Hp,Hc:', (Hp, Hc))

    # Reverse scaling for RL inputs
    u_rl = reverse_min_max(u_mpc, data_min[:n_inputs], data_max[:n_inputs])

    return (y_system, u_rl, avg_rewards, rewards, xhatdhat, nFE, time_in_sub_episodes,
            y_sp, yhat,
            horizon_trace)

In [28]:
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
(y_system, u_rl, avg_rewards, rewards, xhatdhat, nFE, time_in_sub_episodes,
y_sp, yhat,
horizon_trace) = run_dqn_mpc_horizon_supervisor(cstr, y_sp_scenario, n_tests, set_points_len,
                 steady_states, min_max_dict, dqn_agent, A_aug, B_aug, C_aug,
                 Q1_penalty, Q2_penalty, R1_penalty, R2_penalty, L,
                 data_min, data_max, warm_start, TEST_CYCLE,
                 HORIZON_RECIPES,
                 nominal_qi, nominal_qs, nominal_hA,
                 qi_change, qs_change, ha_change,
                 b_min, b_max, reward_fn, mode="disturb", predict_h=predict_h, cont_h=cont_h)

Sub_Episode: 1 | avg. reward: -3.3678030104287755 | Hp,Hc: (9, 3)
Sub_Episode: 2 | avg. reward: -4.4180211755554195 | Hp,Hc: (9, 3)
Sub_Episode: 3 | avg. reward: -4.413181597389634 | Hp,Hc: (9, 3)
Sub_Episode: 4 | avg. reward: -4.4095009901533055 | Hp,Hc: (9, 3)
Sub_Episode: 5 | avg. reward: -4.406010845118631 | Hp,Hc: (9, 3)
Sub_Episode: 6 | avg. reward: -3.0697371557228155 | Hp,Hc: (8, 4)
Sub_Episode: 7 | avg. reward: -3.5603114290039035 | Hp,Hc: (13, 7)
Sub_Episode: 8 | avg. reward: -2.989962504508095 | Hp,Hc: (19, 6)
Sub_Episode: 9 | avg. reward: -2.6825225632563003 | Hp,Hc: (17, 4)
Sub_Episode: 10 | avg. reward: -2.8163650612206994 | Hp,Hc: (8, 8)
Sub_Episode: 11 | avg. reward: -3.2388259039826948 | Hp,Hc: (8, 3)
Sub_Episode: 12 | avg. reward: -3.2759178080656697 | Hp,Hc: (19, 9)
Sub_Episode: 13 | avg. reward: -2.939638297175764 | Hp,Hc: (15, 5)
Sub_Episode: 14 | avg. reward: -3.3998446357322645 | Hp,Hc: (16, 4)
Sub_Episode: 15 | avg. reward: -3.1226504181734436 | Hp,Hc: (18, 5)
S

In [29]:
from utils.plotting import plot_rl_results_dqn, compare_mpc_rl_nominal_from_dirs

In [35]:
out_dir_rl = plot_rl_results_dqn(
    y_sp=y_sp,
    steady_states=steady_states,
    nFE=nFE,
    delta_t=delta_t,
    time_in_sub_episodes=time_in_sub_episodes,
    y_mpc=y_system,
    u_mpc=u_rl,
    avg_rewards=avg_rewards,
    data_min=data_min,
    data_max=data_max,
    reward_fn=reward_fn,
    horizon_trace=horizon_trace,
    mpc_horizons=(predict_h, cont_h),
    start_episode=3,
    prefix_name="horizon_disturb",
    directory=dir_path
)

In [36]:
out_dir_cmp = compare_mpc_rl_nominal_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=r"C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Data\mpc_results_dist.pickle",
    reward_fn=reward_fn,
    directory=r"Result",
    prefix_name="disturb_compare_horizon",
    start_episode=2
)